# Trained-model analysis: predictive performance, safety, and symbolic rules

This notebook applies one consistent protocol to three already trained classifiers: Decision Tree, XGBoost, and a Multilayer Perceptron (MLP). It does not retrain the estimators or modify their stored decision thresholds.

The analysis addresses four separate questions:

1. how each thresholded model performs against the historical test outcomes;
2. whether the deceased-patient safety rule is correctly enforced;
3. how faithfully a compact PSyKE/CART surrogate reproduces black-box decisions;
4. how strongly recurrent features contribute to the extracted explanations.

Task performance, symbolic fidelity, and safety compliance are different properties. A rule theory may faithfully imitate an inaccurate model, while high task accuracy may be caused by class imbalance. The notebook therefore reports the confusion matrix and class-conditional fidelity in addition to aggregate scores.

The executable theory remains in the standardized model space. A second reporting table converts numerical thresholds back to their pre-standardization units without adding semantic descriptions for identifier codes.

## 0. Reproducibility and required artifacts

Run the notebook from top to bottom with **Run All** in the documented Python 3.11 environment. The project must contain:

```text
data/processed/
  X_train.*
  X_test.*
  y_train.*
  y_test.*

data/
  standard_scaler.joblib

models/
  decision_tree_best.joblib
  xgboost_best.joblib
  mlp_best.joblib
  optimal_thresholds.joblib
```

CSV, Parquet, pandas pickle, and Joblib inputs are accepted. `X_train` is used only as the symbolic extraction pool: PSyKE queries the trained models on these rows. `X_test` and `y_test` are reserved for final evaluation. No new data split or test-set threshold selection is performed here.

In [8]:
import warnings
warnings.filterwarnings('ignore')

In [16]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import importlib.metadata as metadata
import json
import platform
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display


def locate_project_root(start: Path) -> Path:
    """Find the project root without changing the working directory."""

    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        package = candidate / "src" / "trustworthy_cds"
        if package.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate a project root containing src/trustworthy_cds. "
        "Open this notebook from the project directory or its notebooks folder."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.trustworthy_cds.evaluation import (
    binary_classification_metrics,
    class_conditional_fidelity_table,
    confusion_table,
    symbolic_surrogate_metrics,
    symbolic_task_metrics_on_recognized,
)

from src.trustworthy_cds.modeling import (
    ThresholdedClassifier,
    load_predictor,
    positive_class_probability,
)

from src.trustworthy_cds.safety import (
    DeceasedStatusDetector,
    SafetyWrapper,
    assert_safety_invariants,
    safety_impact_summary,
)
from src.trustworthy_cds.symbolic import (
    PrecomputedLabelOracle,
    compact_rule_table,
    evaluate_symbolic_extraction,
    extract_psyke_cart,
    save_psyke_theory,
    selected_rule_features,
)

print("Project root:", PROJECT_ROOT)
print("Python:", platform.python_version())

if platform.python_version_tuple()[:2] != ("3", "11"):
    raise RuntimeError(
        "PSyKE 1.0.4 requires the documented Python 3.11 environment."
    )


Project root: C:\Users\ACER-PC\Desktop\Projects\PythonProjects\trustworthy_clinical_cds
Python: 3.11.1


### Environment check

The stored execution used Python 3.11.1, as required by PSyKE 1.0.4. pandas and scikit-learn versions are printed to make the execution environment auditable.

## 1. Experiment configuration

Decision thresholds are loaded from `optimal_thresholds.joblib` and treated as immutable model inputs. Every complete PSyKE extraction uses the same complexity limits (`max_depth=4`, `max_leaves=12`, and at most 8,000 extraction rows), making the primary cross-model comparison consistent.

In [5]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
NORMALIZATION_DIR = PROJECT_ROOT / "data" 
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_ROOT = PROJECT_ROOT / "reports" / "safety_symbolic_model_analysis"
THEORY_ROOT = PROJECT_ROOT / "theories" / "safety_symbolic_model_analysis"

for directory in (OUTPUT_ROOT, THEORY_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_COLUMN = "readmitted_binary"
REQUIRE_EXACT_DENORMALIZATION = True
REQUIRE_DECEASED_EXAMPLES = True

PSYKE_CONFIG = {
    "max_samples": 8_000,
    "max_depth": 4,
    "max_leaves": 12,
    "random_state": RANDOM_STATE,
}

optimal_thresholds = joblib.load(
    MODELS_DIR / "optimal_thresholds.joblib"
)

MODEL_SPECS = {
    "decision_tree": {
        "label": "Decision Tree",
        "path": MODELS_DIR / "decision_tree_best.joblib",
        "threshold": float(optimal_thresholds["Decision Tree"]),
    },
    "xgboost": {
        "label": "XGBoost",
        "path": MODELS_DIR / "xgboost_best.joblib",
        "threshold": float(optimal_thresholds["XGBoost"]),
    },
    "mlp": {
        "label": "MLP",
        "path": MODELS_DIR / "mlp_best.joblib",
        "threshold": float(optimal_thresholds["MLP"]),
    },
}

display(
    pd.DataFrame(
        [
            {
                "model_key": key,
                "model": spec["label"],
                "path": str(spec["path"]),
                "threshold": spec["threshold"],
            }
            for key, spec in MODEL_SPECS.items()
        ]
    ).set_index("model_key")
)


,model,path,threshold
model_key,,,
decision_tree,Decision Tree,C:\Users\ACER-PC\Desktop\Projects\PythonProjec...,0.500000
xgboost,XGBoost,C:\Users\ACER-PC\Desktop\Projects\PythonProjec...,0.500000
mlp,MLP,C:\Users\ACER-PC\Desktop\Projects\PythonProjec...,0.168384


### Stored threshold configuration

In the stored execution, Decision Tree and XGBoost use a 0.50 threshold, while the MLP uses approximately 0.168. These values directly affect the proportion of positive predictions and must be considered part of the decision system being explained.

## 2. Load the prepared train and test partitions

The loader does not recompute the split. It checks that train and test contain the same columns in the same order, converts boolean indicators to 0/1, and rejects missing, non-numeric, or non-finite feature values. The target may be stored separately or embedded temporarily under `readmitted_binary`.

In [6]:
SUPPORTED_SUFFIXES = (".parquet", ".pq", ".csv", ".pkl", ".pickle", ".joblib")


def resolve_artifact(stems: tuple[str, ...], *, required: bool = True) -> Path | None:
    """Resolve one case-insensitive artifact stem inside DATA_DIR."""

    if not DATA_DIR.exists():
        raise FileNotFoundError(f"Data directory does not exist: {DATA_DIR}")

    accepted = {stem.casefold() for stem in stems}
    matches = sorted(
        path
        for path in DATA_DIR.iterdir()
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_SUFFIXES
        and path.stem.casefold() in accepted
    )
    if len(matches) > 1:
        raise ValueError(
            f"Ambiguous artifacts for {stems}: {[path.name for path in matches]}. "
            "Keep only one supported representation."
        )
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(
            f"Missing artifact in {DATA_DIR}. Expected one of stems {stems} "
            f"with a supported suffix {SUPPORTED_SUFFIXES}."
        )
    return None


def load_serialized(path: Path):
    """Load a locally trusted dataframe, series, or preprocessing artifact."""

    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if suffix == ".joblib":
        # Joblib/pickle files can execute code. Load only project-owned artifacts.
        return joblib.load(path)
    raise ValueError(f"Unsupported artifact format: {path}")


def remove_export_index(frame: pd.DataFrame) -> pd.DataFrame:
    """Remove a conventional dataframe index accidentally exported to CSV."""

    frame = frame.copy()
    if len(frame.columns):
        first = str(frame.columns[0])
        if first.startswith("Unnamed:") or first in {"index", "row_index"}:
            if frame.iloc[:, 0].is_unique:
                frame = frame.drop(columns=frame.columns[0])
    return frame


def load_feature_matrix(path: Path) -> tuple[pd.DataFrame, pd.Series | None]:
    payload = load_serialized(path)
    if not isinstance(payload, pd.DataFrame):
        raise TypeError(
            f"{path.name} must contain a pandas DataFrame so feature names are preserved."
        )
    frame = remove_export_index(payload)
    embedded_target = None
    if TARGET_COLUMN in frame:
        embedded_target = pd.to_numeric(frame.pop(TARGET_COLUMN), errors="raise")
    if frame.empty or frame.columns.has_duplicates:
        raise ValueError(f"Invalid or empty feature matrix: {path.name}")
    boolean_columns = frame.select_dtypes(include=["bool"]).columns
    if len(boolean_columns):
        frame[boolean_columns] = frame[boolean_columns].astype(np.uint8)
    non_numeric = frame.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        raise TypeError(f"Non-numeric columns in {path.name}: {non_numeric}")
    frame = frame.astype(float).reset_index(drop=True)
    if frame.isna().any().any() or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"Missing or non-finite feature values in {path.name}.")
    if embedded_target is not None:
        embedded_target = embedded_target.reset_index(drop=True)
    return frame, embedded_target


def load_target(path: Path | None, embedded: pd.Series | None, name: str) -> pd.Series | None:
    if path is None:
        target = embedded
    else:
        payload = load_serialized(path)
        if isinstance(payload, pd.DataFrame):
            payload = remove_export_index(payload)
            preferred = [column for column in payload.columns if str(column) == TARGET_COLUMN]
            if preferred:
                payload = payload[preferred[0]]
            elif payload.shape[1] == 1:
                payload = payload.iloc[:, 0]
            else:
                raise ValueError(
                    f"{path.name} contains multiple columns and none is {TARGET_COLUMN!r}."
                )
        target = pd.Series(payload)
    if target is None:
        return None
    target = pd.to_numeric(target, errors="raise").astype(int).reset_index(drop=True)
    observed = set(target.unique().tolist())
    if not observed.issubset({0, 1}):
        raise ValueError(f"{name} must contain only 0/1 labels, found {sorted(observed)}.")
    return target


SPLIT_STEMS = {
    "train": ("X_train", "x_train"),
    "test": ("X_test", "x_test"),
}
TARGET_STEMS = {
    "train": ("y_train", "Y_train"),
    "test": ("y_test", "Y_test"),
}

split_frames = {}
split_targets = {}
split_paths = {}

for split_name in ("train", "test"):
    x_path = resolve_artifact(SPLIT_STEMS[split_name])
    X_split, embedded_y = load_feature_matrix(x_path)
    y_path = resolve_artifact(TARGET_STEMS[split_name], required=False)
    y_split = load_target(y_path, embedded_y, f"y_{split_name}")

    if y_split is not None and len(y_split) != len(X_split):
        raise ValueError(
            f"X_{split_name} and y_{split_name} have different row counts."
        )
    if split_name == "test" and y_split is None:
        raise FileNotFoundError(
            "y_test is required for final task-performance metrics. "
            "Save it separately or include readmitted_binary in X_test."
        )

    split_frames[split_name] = X_split
    split_targets[split_name] = y_split
    split_paths[split_name] = {"X": x_path, "y": y_path}

reference_columns = list(split_frames["train"].columns)
for split_name, frame in split_frames.items():
    if list(frame.columns) != reference_columns:
        missing = sorted(set(reference_columns) - set(frame.columns))
        extra = sorted(set(frame.columns) - set(reference_columns))
        raise ValueError(
            f"Feature contract mismatch in X_{split_name}. "
            f"Missing={missing}; extra={extra}; order must also match."
        )

split_summary = []
for split_name in ("train", "test"):
    target = split_targets[split_name]
    split_summary.append(
        {
            "partition": split_name,
            "rows": len(split_frames[split_name]),
            "columns_in_file": split_frames[split_name].shape[1],
            "target_available": target is not None,
            "positive_rate": float(target.mean()) if target is not None else np.nan,
            "X_artifact": split_paths[split_name]["X"].name,
            "y_artifact": (
                split_paths[split_name]["y"].name
                if split_paths[split_name]["y"] is not None
                else "embedded or unavailable"
            ),
        }
    )

display(pd.DataFrame(split_summary).set_index("partition"))


,rows,columns_in_file,target_available,positive_rate,X_artifact,y_artifact
partition,,,,,,
train,57204,148,True,0.087983,X_train.joblib,y_train.joblib
test,14302,148,True,0.087960,X_test.joblib,y_test.joblib


### Data distribution

The stored run contains 57,204 training rows and 14,302 test rows, each with 148 features. Positive prevalence is nearly identical in both partitions, at approximately 8.8%, but the target is strongly imbalanced.

Consequently, an always-`not_readmitted` dummy rule obtains approximately 91.2% task accuracy while having zero recall for `readmitted`. Accuracy alone is therefore insufficient.

## 3. Load and validate the trained models

In [9]:
loaded_models = {}
model_contract = None
model_overview = []

for model_key, spec in MODEL_SPECS.items():
    try:
        model = load_predictor(spec["path"])
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            f"Could not load {spec['label']}. Install the package named in the "
            f"original error ({exc.name!r}) inside the Python 3.11 environment."
        ) from exc

    for method in ("predict", "predict_proba"):
        if not hasattr(model, method):
            raise TypeError(f"{spec['label']} does not expose {method}().")
    if not hasattr(model, "classes_"):
        raise TypeError(f"{spec['label']} does not expose classes_.")
    if set(np.asarray(model.classes_).tolist()) != {0, 1}:
        raise ValueError(f"{spec['label']} must contain exactly classes 0 and 1.")

    names = getattr(model, "feature_names_in_", None)
    if names is None:
        raise ValueError(
            f"{spec['label']} has no feature_names_in_. Exact feature validation "
            "is required before a clinical analysis."
        )
    names = tuple(str(name) for name in names)

    if model_contract is None:
        model_contract = names
    elif names != model_contract:
        raise ValueError(
            f"{spec['label']} uses a different ordered feature contract from the "
            "other trained models. Run separate analyses or retrain consistently."
        )

    if "patient_count" in names:
        raise ValueError(
            f"{spec['label']} still contains patient_count, which is forbidden by "
            "the current temporal feature contract."
        )

    missing_by_split = {
        split_name: sorted(set(names) - set(frame.columns))
        for split_name, frame in split_frames.items()
    }
    missing_by_split = {
        split_name: missing
        for split_name, missing in missing_by_split.items()
        if missing
    }
    if missing_by_split:
        raise ValueError(
            f"{spec['label']} model features are missing from the split files: "
            f"{missing_by_split}"
        )

    loaded_models[model_key] = model
    model_overview.append(
        {
            "model": spec["label"],
            "estimator_type": type(model).__name__,
            "n_model_features": len(names),
            "n_discharge_features": sum(
                name.startswith("discharge_disposition_id_") for name in names
            ),
            "threshold": spec["threshold"],
        }
    )

assert model_contract is not None
display(pd.DataFrame(model_overview).set_index("model"))
print("Shared ordered feature contract:", len(model_contract), "features")


,estimator_type,n_model_features,n_discharge_features,threshold
model,,,,
Decision Tree,DecisionTreeClassifier,148,26,0.500000
XGBoost,XGBClassifier,148,26,0.500000
MLP,MLPClassifier,148,26,0.168384


Shared ordered feature contract: 148 features


### Feature contract and version compatibility

The three estimators share an ordered contract of 148 features, including 26 one-hot discharge-disposition variables. Their symbolic extractions are therefore technically comparable.

## 4. Inverse transformation of rule thresholds

For a standardized feature, a CART threshold `z` is returned to its pre-standardization scale through:

\[
x = z \cdot \mathrm{scale} + \mathrm{mean}.
\]

The exact fitted mean and scale are required and cannot be reconstructed uniquely from already standardized data. The preferred artifact is `data/standard_scaler.joblib`. The Prolog theory stays in model space; the denormalized table is a reporting view.

In [10]:
STANDARDIZED_FEATURES = (
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age",
    "max_glu_serum",
    "A1Cresult",
)

POST_STANDARDIZATION_INVERSES = {
    "num_medications": "expm1",
}

FEATURE_DISPLAY_NAMES = {
    "time_in_hospital": "time in hospital",
    "num_lab_procedures": "number of laboratory procedures",
    "num_procedures": "number of procedures",
    "number_diagnoses": "number of diagnoses (grouped)",
    "num_medications": "number of medications",
    "number_outpatient": "previous outpatient visits (binned)",
    "number_emergency": "previous emergency visits (binned)",
    "number_inpatient": "previous inpatient visits (binned)",
    "age": "age band",
    "max_glu_serum": "maximum glucose serum result",
    "A1Cresult": "A1C result",
}

FEATURE_UNITS = {
    "time_in_hospital": "days",
}

DISCRETE_VALUE_LABELS = {
    "number_outpatient": {0: "0", 1: "1", 2: "2", 3: "3 or more"},
    "number_emergency": {0: "0", 1: "1", 2: "2", 3: "3 or more"},
    "number_inpatient": {0: "0", 1: "1", 2: "2", 3: "3 or more"},
    "age": {
        0: "[0-10)", 1: "[10-20)", 2: "[20-30)", 3: "[30-40)",
        4: "[40-50)", 5: "[50-60)", 6: "[60-70)", 7: "[70-80)",
        8: "[80-90)", 9: "[90-100)",
    },
    "max_glu_serum": {
        0: "not measured", 1: "normal", 2: ">200", 3: ">300",
    },
    "A1Cresult": {
        0: "not measured", 1: "normal", 2: ">7", 3: ">8",
    },
    "number_diagnoses": {
        **{value: str(value) for value in range(1, 10)},
        10: "10 or more",
    },
}


def scaler_to_registry(scaler, *, source: str) -> dict[str, dict[str, object]]:
    """Convert a fitted StandardScaler into per-feature inverse parameters."""

    if not hasattr(scaler, "mean_") or not hasattr(scaler, "scale_"):
        raise TypeError("The normalization artifact is not a fitted StandardScaler.")
    names = getattr(scaler, "feature_names_in_", None)
    if names is None:
        if len(scaler.mean_) != len(STANDARDIZED_FEATURES):
            raise ValueError(
                "Scaler has no feature_names_in_ and its length does not match the "
                "documented standardized feature list."
            )
        names = STANDARDIZED_FEATURES
    names = [str(name) for name in names]
    if len(names) != len(scaler.mean_) or len(names) != len(scaler.scale_):
        raise ValueError("Scaler feature names, means, and scales are inconsistent.")

    registry = {}
    for feature, mean, scale in zip(names, scaler.mean_, scaler.scale_):
        scale = float(scale)
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError(f"Invalid scale for {feature!r}: {scale}")
        registry[feature] = {
            "mean": float(mean),
            "scale": scale,
            "post_inverse": POST_STANDARDIZATION_INVERSES.get(feature, "identity"),
            "source": source,
        }
    return registry


def metadata_to_registry(payload, *, source: str) -> dict[str, dict[str, object]]:
    """Read a JSON/dict registry with exact mean and scale per feature."""

    if isinstance(payload, dict) and "scaler" in payload:
        return scaler_to_registry(payload["scaler"], source=source)
    if isinstance(payload, dict) and "features" in payload:
        payload = payload["features"]
    if not isinstance(payload, dict):
        raise TypeError("Normalization metadata must be a scaler or feature dictionary.")

    registry = {}
    for feature, spec in payload.items():
        if not isinstance(spec, dict) or not {"mean", "scale"}.issubset(spec):
            raise ValueError(
                f"Normalization metadata for {feature!r} needs mean and scale."
            )
        scale = float(spec["scale"])
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError(f"Invalid scale for {feature!r}: {scale}")
        registry[str(feature)] = {
            "mean": float(spec["mean"]),
            "scale": scale,
            "post_inverse": spec.get(
                "post_inverse",
                POST_STANDARDIZATION_INVERSES.get(str(feature), "identity"),
            ),
            "source": source,
        }
    return registry


def infer_registry_from_paired_frames(
    standardized: pd.DataFrame,
    prestandardized: pd.DataFrame,
    features: tuple[str, ...],
    *,
    source: str,
) -> dict[str, dict[str, object]]:
    """Infer raw = standardized * scale + mean from row-aligned paired data."""

    if len(standardized) != len(prestandardized):
        raise ValueError("Paired standardized and pre-standardization rows differ.")
    registry = {}
    for feature in features:
        if feature not in standardized or feature not in prestandardized:
            raise KeyError(f"Paired normalization data is missing {feature!r}.")
        z = standardized[feature].to_numpy(dtype=float)
        x = prestandardized[feature].to_numpy(dtype=float)
        if np.std(z) == 0:
            raise ValueError(f"Cannot infer scaling for constant feature {feature!r}.")
        scale, mean = np.polyfit(z, x, deg=1)
        reconstructed = z * scale + mean
        if scale <= 0 or not np.allclose(
            reconstructed, x, rtol=1e-6, atol=1e-7
        ):
            raise ValueError(
                f"{feature!r} is not related by a valid StandardScaler affine map."
            )
        registry[feature] = {
            "mean": float(mean),
            "scale": float(scale),
            "post_inverse": POST_STANDARDIZATION_INVERSES.get(feature, "identity"),
            "source": source,
        }
    return registry


def load_normalization_registry() -> tuple[dict[str, dict[str, object]], str]:
    """Load exact inverse parameters, preferring the fitted scaler artifact."""

    preferred_names = (
        "standard_scaler.joblib",
        "preprocessing_scaler.joblib",
        "scaler.joblib",
        "normalization_metadata.joblib",
        "normalization_metadata.json",
    )
    for filename in preferred_names:
        path = NORMALIZATION_DIR / filename
        if not path.exists():
            continue
        if path.suffix.lower() == ".json":
            payload = json.loads(path.read_text(encoding="utf-8"))
        else:
            payload = joblib.load(path)
        if hasattr(payload, "mean_") and hasattr(payload, "scale_"):
            return scaler_to_registry(payload, source=path.name), path.name
        return metadata_to_registry(payload, source=path.name), path.name

    paired_path = resolve_artifact(
        (
            "X_train_prestandardization",
            "x_train_prestandardization",
            "X_train_unscaled",
            "x_train_unscaled",
        ),
        required=False,
    )
    if paired_path is not None:
        paired_frame, _ = load_feature_matrix(paired_path)
        registry = infer_registry_from_paired_frames(
            split_frames["train"],
            paired_frame,
            tuple(
                feature
                for feature in STANDARDIZED_FEATURES
                if feature in model_contract
            ),
            source=paired_path.name,
        )
        return registry, paired_path.name

    message = (
        "Exact denormalization is unavailable. Save the fitted preprocessing "
        "StandardScaler as data/standard_scaler.joblib, or provide a row-aligned "
        "X_train_prestandardization artifact. Standardized values alone do not "
        "uniquely determine the original units."
    )
    if REQUIRE_EXACT_DENORMALIZATION:
        raise FileNotFoundError(message)
    warnings.warn(message)
    return {}, "unavailable"


normalization_registry, normalization_source = load_normalization_registry()

required_inverse_features = {
    feature for feature in STANDARDIZED_FEATURES if feature in model_contract
}
missing_inverse_features = sorted(
    required_inverse_features - set(normalization_registry)
)
if missing_inverse_features and REQUIRE_EXACT_DENORMALIZATION:
    raise ValueError(
        "Normalization metadata is incomplete for standardized model features: "
        f"{missing_inverse_features}"
    )

normalization_table = pd.DataFrame.from_dict(
    normalization_registry,
    orient="index",
).rename_axis("feature")
display(normalization_table)
print("Normalization source:", normalization_source)


,mean,scale,post_inverse,source
feature,,,,
time_in_hospital,4.294000,2.956250,identity,standard_scaler.joblib
num_lab_procedures,43.057286,19.955774,identity,standard_scaler.joblib
num_procedures,1.431299,1.760476,identity,standard_scaler.joblib
num_medications,2.696376,0.504765,expm1,standard_scaler.joblib
number_outpatient,0.226837,0.660801,identity,standard_scaler.joblib
number_emergency,0.096514,0.384024,identity,standard_scaler.joblib
number_inpatient,0.166474,0.511316,identity,standard_scaler.joblib
number_diagnoses,7.244389,1.978511,identity,standard_scaler.joblib
age,6.065765,1.592357,identity,standard_scaler.joblib


Normalization source: standard_scaler.joblib


### Loaded inverse parameters

The registry confirms that the eleven transformed numerical features are linked to the fitted `standard_scaler.joblib`. Thresholds for discrete features such as `number_inpatient` can therefore be rendered as the documented pre-standardization categories rather than as opaque z-scores.

### Human-readable rule construction

The following functions reconstruct CART paths, merge repeated tests on the same feature, and apply numerical inverse transformations where available. One-hot features retain their identifier codes;

In [11]:
ONE_HOT_PREFIX_LABELS = (
    ("race_", "race"),
    ("admission_type_id_", "admission type code"),
    ("discharge_disposition_id_", "discharge disposition code"),
    ("admission_source_id_", "admission source code"),
    ("payer_code_", "payer code"),
    ("medical_specialty_", "medical specialty"),
    ("diag_1_", "primary diagnosis group"),
    ("diag_2_", "secondary diagnosis group"),
    ("diag_3_", "tertiary diagnosis group"),
)


def inverse_feature_value(
    feature: str,
    standardized_value: float,
    registry: dict[str, dict[str, object]],
) -> float:
    """Inverse-transform one tree threshold for any registered feature."""

    if feature not in registry:
        return float(standardized_value)
    spec = registry[feature]
    value = float(standardized_value) * float(spec["scale"]) + float(spec["mean"])
    post_inverse = spec.get("post_inverse", "identity")
    if post_inverse == "expm1":
        value = float(np.expm1(value))
    elif post_inverse != "identity":
        raise ValueError(
            f"Unsupported post-standardization inverse {post_inverse!r} "
            f"for {feature!r}."
        )
    return value


def clean_threshold(value: float) -> float:
    """Remove floating-point noise while retaining real CART boundaries."""

    for step in (0.5, 1.0):
        snapped = round(value / step) * step
        if np.isclose(value, snapped, atol=1e-6, rtol=0):
            return float(snapped)
    return float(value)


def format_threshold(value: float) -> str:
    value = clean_threshold(value)
    if np.isclose(value, round(value), atol=1e-8, rtol=0):
        return str(int(round(value)))
    return f"{value:.4g}"


def readable_feature_name(feature: str) -> str:
    return FEATURE_DISPLAY_NAMES.get(feature, feature.replace("_", " "))


def render_binary_condition(feature: str, operator: str, threshold: float) -> str | None:
    if not np.isclose(threshold, 0.5, atol=0.05):
        return None
    present = operator == ">"
    for prefix, label in ONE_HOT_PREFIX_LABELS:
        if feature.startswith(prefix):
            category = feature[len(prefix):].replace("_", " ")
            return f"{label} {'is' if present else 'is not'} {category}"
    return f"{readable_feature_name(feature)} is {'true' if present else 'false'}"


def consolidate_path(path: list[tuple[str, str, float]]):
    order = []
    bounds = {}
    for feature, operator, threshold in path:
        if feature not in bounds:
            order.append(feature)
            bounds[feature] = {"lower": None, "upper": None}
        if operator == ">":
            previous = bounds[feature]["lower"]
            bounds[feature]["lower"] = (
                threshold if previous is None else max(previous, threshold)
            )
        elif operator == "<=":
            previous = bounds[feature]["upper"]
            bounds[feature]["upper"] = (
                threshold if previous is None else min(previous, threshold)
            )
        else:
            raise ValueError(f"Unsupported CART operator: {operator}")
    return [
        (feature, bounds[feature]["lower"], bounds[feature]["upper"])
        for feature in order
    ]


def render_discrete_interval(
    feature: str,
    lower: float | None,
    upper: float | None,
) -> str | None:
    labels = DISCRETE_VALUE_LABELS.get(feature)
    if labels is None:
        return None
    allowed = [
        value
        for value in sorted(labels)
        if (lower is None or value > lower)
        and (upper is None or value <= upper)
    ]
    if not allowed:
        return None
    rendered = [labels[value] for value in allowed]
    name = readable_feature_name(feature)
    if len(rendered) == 1:
        return f"{name} is {rendered[0]}"
    return f"{name} is one of {{{', '.join(rendered)}}}"


def render_denormalized_condition(
    feature: str,
    lower: float | None,
    upper: float | None,
    *,
    binary_features: set[str],
    registry: dict[str, dict[str, object]],
) -> str:
    was_standardized = feature in registry
    raw_lower = (
        clean_threshold(inverse_feature_value(feature, lower, registry))
        if lower is not None
        else None
    )
    raw_upper = (
        clean_threshold(inverse_feature_value(feature, upper, registry))
        if upper is not None
        else None
    )

    if was_standardized:
        discrete = render_discrete_interval(feature, raw_lower, raw_upper)
        if discrete is not None:
            return discrete

    if not was_standardized and feature in binary_features:
        if raw_lower is not None and raw_upper is None:
            rendered = render_binary_condition(feature, ">", raw_lower)
            if rendered is not None:
                return rendered
        if raw_upper is not None and raw_lower is None:
            rendered = render_binary_condition(feature, "<=", raw_upper)
            if rendered is not None:
                return rendered

    name = readable_feature_name(feature)
    unit = FEATURE_UNITS.get(feature, "")
    suffix = f" {unit}" if unit else ""
    if raw_lower is not None and raw_upper is not None:
        return (
            f"{format_threshold(raw_lower)} < {name} <= "
            f"{format_threshold(raw_upper)}{suffix}"
        )
    if raw_lower is not None:
        return f"{name} > {format_threshold(raw_lower)}{suffix}"
    if raw_upper is not None:
        return f"{name} <= {format_threshold(raw_upper)}{suffix}"
    return "always"


def denormalized_rule_table(symbolic, registry) -> pd.DataFrame:
    """Create a reporting-only rule table with original-unit thresholds."""

    normalized_rules = compact_rule_table(symbolic).copy()
    tree_predictor = symbolic.extractor._cart_predictor.predictor
    tree = tree_predictor.tree_
    binary_features = set(symbolic.binary_original_features)
    rendered_paths = []
    transformed_features = []

    def visit(node: int, path: list[tuple[str, str, float]]) -> None:
        left = int(tree.children_left[node])
        right = int(tree.children_right[node])
        if left == right:
            conditions = []
            transformed = []
            for feature, lower, upper in consolidate_path(path):
                conditions.append(
                    render_denormalized_condition(
                        feature,
                        lower,
                        upper,
                        binary_features=binary_features,
                        registry=registry,
                    )
                )
                if feature in registry:
                    transformed.append(feature)
            rendered_paths.append(" AND ".join(conditions) if conditions else "always")
            transformed_features.append(", ".join(transformed) or "not required")
            return

        safe_feature = str(tree_predictor.feature_names_in_[tree.feature[node]])
        feature = symbolic.feature_map.safe_to_original[safe_feature]
        threshold = float(tree.threshold[node])
        visit(left, [*path, (feature, "<=", threshold)])
        visit(right, [*path, (feature, ">", threshold)])

    visit(0, [])
    if len(rendered_paths) != len(normalized_rules):
        raise RuntimeError("Denormalized paths do not align with compact CART rules.")
    normalized_rules.insert(2, "if_denormalized", rendered_paths)
    normalized_rules.insert(3, "denormalized_features", transformed_features)
    return normalized_rules


# A deterministic sanity check for the generic inverse formula.
_demo_registry = {
    "demo": {"mean": 10.0, "scale": 2.0, "post_inverse": "identity"}
}
assert np.isclose(inverse_feature_value("demo", -0.5, _demo_registry), 9.0)
print("Denormalization helpers validated.")


Denormalization helpers validated.


## 5. Build the safety-verification context

Safety verification is separate from predictive evaluation. The context combines the test set with every historical row recognized as deceased in the supplied partitions. 

In [13]:
DECEASED_COLUMNS = (
    "discharge_disposition_id_11",
    "discharge_disposition_id_19",
    "discharge_disposition_id_20",
    "discharge_disposition_id_21",
)


def tagged_context(frame: pd.DataFrame, partition: str) -> pd.DataFrame:
    tagged = frame.copy()
    tagged.index = pd.Index(
        [f"{partition}:{position}" for position in range(len(tagged))],
        name="source_row",
    )
    return tagged


safety_path = resolve_artifact(
    (
        "safety_context",
        "X_safety",
        "x_safety",
        "X_test_context",
        "x_test_context",
    ),
    required=False,
)

if safety_path is not None:
    safety_evaluation_context, _ = load_feature_matrix(safety_path)
    safety_evaluation_context = tagged_context(
        safety_evaluation_context,
        "safety",
    )
    safety_context_source = safety_path.name
else:
    tagged_splits = {
        split_name: tagged_context(frame, split_name)
        for split_name, frame in split_frames.items()
    }
    all_supplied_rows = pd.concat(
        [tagged_splits[name] for name in ("train", "test")],
        axis=0,
    )
    detector_for_selection = DeceasedStatusDetector(
        deceased_columns=DECEASED_COLUMNS,
    )
    try:
        deceased_across_splits = detector_for_selection.detect(all_supplied_rows)
    except KeyError as exc:
        raise KeyError(
            "No safety signal is available in the split files. Provide a model-ready "
            "safety_context artifact containing all model features plus is_deceased "
            "or a recognized deceased discharge-disposition dummy."
        ) from exc
    safety_evaluation_context = pd.concat(
        [
            tagged_splits["test"],
            all_supplied_rows.loc[deceased_across_splits],
        ],
        axis=0,
    )
    safety_evaluation_context = safety_evaluation_context.loc[
        ~safety_evaluation_context.index.duplicated(keep="first")
    ]
    safety_context_source = "X_test plus deceased rows from supplied partitions"

detector = DeceasedStatusDetector(deceased_columns=DECEASED_COLUMNS)
detected_deceased = detector.detect(safety_evaluation_context)

missing_safety_model_features = sorted(
    set(model_contract) - set(safety_evaluation_context.columns)
)
if missing_safety_model_features:
    raise ValueError(
        "Safety context is missing model features: "
        f"{missing_safety_model_features}"
    )
if REQUIRE_DECEASED_EXAMPLES and not detected_deceased.any():
    raise ValueError(
        "The safety context contains no deceased encounter. The hard rule cannot be "
        "empirically exercised; provide a safety_context artifact with such cases."
    )

print("Safety context source:", safety_context_source)
display(
    pd.Series(
        {
            "rows checked": len(safety_evaluation_context),
            "deceased rows": int(detected_deceased.sum()),
            "non-deceased rows": int((~detected_deceased).sum()),
        },
        name="value",
    ).to_frame()
)


Safety context source: X_test plus deceased rows from supplied partitions


,value
rows checked,15172
deceased rows,1084
non-deceased rows,14088


### Safety-audit sample

The audit covers 15,172 rows: 1,084 deceased encounters and 14,088 non-deceased encounters. This context is used only to test the wrapper and does not replace the test set used for predictive metrics.

## 6. Shared experiment protocol

For each classifier the notebook:

1. applies the stored threshold and computes compact test metrics;
2. verifies the safety wrapper on the dedicated context;
3. extracts a PSyKE/CART surrogate from black-box training decisions;
4. evaluates coverage, overall fidelity, and class-conditional fidelity on test data;
5. compares PSyKE with a constant rule learned from training predictions;
6. exports metrics, rules, theory, and audit evidence.

Fidelity compares the symbolic theory with the black box. Task metrics compare predictions with `y_test`. They answer different questions and are kept separate.

In [23]:
def json_safe(value):
    """Recursively convert numpy/pandas values to strict JSON values."""

    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        number = float(value)
        return number if np.isfinite(number) else None
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(payload, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(json_safe(payload), indent=2, allow_nan=False),
        encoding="utf-8",
    )


def run_model_experiment(model_key: str) -> dict[str, object]:
    spec = MODEL_SPECS[model_key]
    raw_model = loaded_models[model_key]
    predictor = ThresholdedClassifier(
        raw_model,
        threshold=float(spec["threshold"]),
    )

    X_train_model = split_frames["train"].loc[:, model_contract]
    X_test_model = split_frames["test"].loc[:, model_contract]
    y_test = split_targets["test"]
    assert y_test is not None

    report_dir = OUTPUT_ROOT / model_key
    theory_dir = THEORY_ROOT / model_key
    report_dir.mkdir(parents=True, exist_ok=True)
    theory_dir.mkdir(parents=True, exist_ok=True)

    # 1. Black-box task performance on the untouched test partition.
    test_probability = positive_class_probability(predictor, X_test_model)
    test_prediction = predictor.predict(X_test_model)
    all_test_metrics = binary_classification_metrics(
        y_test,
        test_prediction,
        test_probability,
    )
    confusion = confusion_table(all_test_metrics)
    reported_metric_keys = (
        "n_samples",
        "prevalence",
        "predicted_positive_rate",
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall_sensitivity",
        "specificity",
        "f1",
        "f2",
        "roc_auc",
        "average_precision",
    )
    test_metrics = {
        key: all_test_metrics[key]
        for key in reported_metric_keys
    }

    # 2. Model-independent deceased-patient safety rule.
    safe_predictor = SafetyWrapper(
        predictor,
        model_features=model_contract,
        detector=detector,
    )
    safety_verification = assert_safety_invariants(
        safe_predictor,
        safety_evaluation_context,
    )
    safety_audit = safe_predictor.predict_with_audit(
        safety_evaluation_context,
    )

    # 3. PSyKE CART learns black-box decisions on training inputs only.
    symbolic = extract_psyke_cart(
        predictor,
        X_train_model,
        **PSYKE_CONFIG,
    )
    symbolic_metrics = evaluate_symbolic_extraction(
        symbolic,
        X_test_model,
        y_test,
    )
    standardized_rules = compact_rule_table(symbolic)
    reporting_rules = denormalized_rule_table(
        symbolic,
        normalization_registry,
    )

    # 4. Overall and class-conditional fidelity on the same test rows.
    train_safe = symbolic.feature_map.to_safe(X_train_model)
    test_safe = symbolic.feature_map.to_safe(X_test_model)
    black_box_train_labels = np.asarray(
        symbolic.oracle.predict(train_safe),
        dtype=object,
    )
    black_box_test_labels = np.asarray(
        symbolic.oracle.predict(test_safe),
        dtype=object,
    )
    symbolic_test_labels = np.asarray(
        symbolic.extractor.predict(test_safe),
        dtype=object,
    )
    classwise = class_conditional_fidelity_table(
        black_box_test_labels,
        symbolic_test_labels,
    )

    # 5. One-rule benchmark learned from training black-box predictions.
    trivial_label = (
        pd.Series(black_box_train_labels).value_counts().idxmax()
    )
    trivial_test_labels = np.full(
        len(black_box_test_labels),
        trivial_label,
        dtype=object,
    )
    trivial_metrics = symbolic_surrogate_metrics(
        black_box_test_labels,
        trivial_test_labels,
        y_test,
    )
    psyke_comparison_metrics = symbolic_surrogate_metrics(
        black_box_test_labels,
        symbolic_test_labels,
        y_test,
    )
    absolute_gain = (
        psyke_comparison_metrics["fidelity_overall"]
        - trivial_metrics["fidelity_overall"]
    )
    remaining = 1.0 - trivial_metrics["fidelity_overall"]
    normalized_gain = absolute_gain / remaining if remaining > 0 else np.nan

    surrogate_comparison = pd.DataFrame(
        [
            {
                "surrogate": "Trivial constant rule",
                "constant_label": trivial_label,
                "coverage": trivial_metrics["coverage"],
                "fidelity_overall": trivial_metrics["fidelity_overall"],
                "task_accuracy_overall": trivial_metrics["task_accuracy_overall"],
                "n_rules": 1,
                "max_rule_length": 0,
            },
            {
                "surrogate": "PSyKE CART",
                "constant_label": "not applicable",
                "coverage": psyke_comparison_metrics["coverage"],
                "fidelity_overall": psyke_comparison_metrics["fidelity_overall"],
                "task_accuracy_overall": psyke_comparison_metrics[
                    "task_accuracy_overall"
                ],
                "n_rules": int(symbolic.n_rules),
                "max_rule_length": int(standardized_rules["rule_length"].max()),
            },
        ]
    ).set_index("surrogate")

    symbolic_task_metrics = symbolic_task_metrics_on_recognized(
        y_test,
        symbolic_test_labels,
    )

    # Add the most important comparison values to the JSON metric bundle.
    symbolic_metrics.update(
        {
            "fidelity_black_box_not_readmitted": classwise.loc[
                "not_readmitted", "class_conditional_fidelity"
            ],
            "fidelity_black_box_readmitted": classwise.loc[
                "readmitted", "class_conditional_fidelity"
            ],
            "n_black_box_not_readmitted": classwise.loc[
                "not_readmitted", "n_black_box_predictions"
            ],
            "n_black_box_readmitted": classwise.loc[
                "readmitted", "n_black_box_predictions"
            ],
            "trivial_fidelity_overall": trivial_metrics["fidelity_overall"],
            "absolute_fidelity_gain_over_trivial": absolute_gain,
            "normalized_fidelity_gain_over_trivial": normalized_gain,
        }
    )

    selected_features = pd.DataFrame(
        {
            "feature": list(selected_rule_features(symbolic)),
        }
    )
    selected_features["standardized"] = selected_features["feature"].isin(
        normalization_registry
    )
    #selected_features["available_at_declared_decision_time"] = ~selected_features[
    #    "feature"
    #].str.startswith("discharge_disposition_id_")

    # 6. Reproducible evidence bundle.
    write_json(test_metrics, report_dir / "test_metrics.json")
    write_json(safety_verification, report_dir / "safety_verification.json")
    write_json(symbolic_metrics, report_dir / "symbolic_metrics.json")
    write_json(symbolic_task_metrics, report_dir / "symbolic_task_metrics.json")
    write_json(
        {
            "model_key": model_key,
            "model_label": spec["label"],
            "model_path": spec["path"],
            "threshold": spec["threshold"],
            "n_model_features": len(model_contract),
            "normalization_source": normalization_source,
            "psyke_config": PSYKE_CONFIG,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "python": platform.python_version(),
            "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__,
        },
        report_dir / "run_manifest.json",
    )
    write_json(
        {
            "original_to_psyke": symbolic.feature_map.original_to_safe,
            "psyke_to_original": symbolic.feature_map.safe_to_original,
        },
        report_dir / "symbolic_feature_name_map.json",
    )

    confusion.to_csv(report_dir / "test_confusion_matrix.csv")
    safety_audit.to_csv(report_dir / "safety_audit.csv", index_label="row_index")
    standardized_rules.to_csv(
        report_dir / "compact_symbolic_rules_standardized.csv",
        index=False,
    )
    reporting_rules.to_csv(
        report_dir / "compact_symbolic_rules_denormalized.csv",
        index=False,
    )
    classwise.to_csv(report_dir / "class_conditional_fidelity.csv")
    surrogate_comparison.to_csv(report_dir / "surrogate_comparison.csv")
    selected_features.to_csv(report_dir / "selected_rule_features.csv", index=False)
    save_psyke_theory(
        symbolic,
        theory_dir / "psyke_cart_theory_standardized.pl",
    )

    summary = {
        "model_key": model_key,
        "model": spec["label"],
        "threshold": spec["threshold"],
        "n_features": len(model_contract),
        "test_recall": test_metrics["recall_sensitivity"],
        "test_precision": test_metrics["precision"],
        "test_f2": test_metrics["f2"],
        "test_roc_auc": test_metrics.get("roc_auc", np.nan),
        "test_average_precision": test_metrics.get("average_precision", np.nan),
        "safety_deceased_rows": safety_verification["n_deceased_rows"],
        "safety_passed": safety_verification["all_invariants_passed"],
        "symbolic_coverage": symbolic_metrics["coverage"],
        "symbolic_fidelity_overall": symbolic_metrics["fidelity_overall"],
        "fidelity_not_readmitted": classwise.loc[
            "not_readmitted", "class_conditional_fidelity"
        ],
        "fidelity_readmitted": classwise.loc[
            "readmitted", "class_conditional_fidelity"
        ],
        "trivial_fidelity": trivial_metrics["fidelity_overall"],
        "fidelity_gain_over_trivial": absolute_gain,
        "n_rules": int(symbolic.n_rules),
        "tree_depth": symbolic_metrics["surrogate_tree_depth"],
        "n_selected_features": symbolic_metrics["n_distinct_features_used"],
    }

    return {
        "summary": summary,
        "test_metrics": test_metrics,
        "confusion": confusion,
        "safety_verification": safety_verification,
        "safety_audit": safety_audit,
        "symbolic_metrics": symbolic_metrics,
        "classwise": classwise,
        "surrogate_comparison": surrogate_comparison,
        "standardized_rules": standardized_rules,
        "reporting_rules": reporting_rules,
        "selected_features": selected_features,
        "report_dir": report_dir,
        "theory_dir": theory_dir,
    }


## 7. Run the three complete experiments

The next cell processes Decision Tree, XGBoost, and MLP under the same protocol. Runtime depends partly on JVM and PSyKE startup.

In [24]:
experiment_results = {}

for model_key, spec in MODEL_SPECS.items():
    print("\n" + "=" * 88)
    print(f"Running complete experiment: {spec['label']}")
    print("=" * 88)

    result = run_model_experiment(model_key)
    experiment_results[model_key] = result

    print("\nBlack-box test metrics")
    display(pd.Series(result["test_metrics"], name="value").to_frame())
    display(result["confusion"])

    print("Safety verification")
    display(pd.Series(result["safety_verification"], name="value").to_frame())

    print("Symbolic extraction metrics")
    display(pd.Series(result["symbolic_metrics"], name="value").to_frame())

    print("Class-conditional fidelity")
    display(result["classwise"])

    print("Trivial rule versus PSyKE")
    display(result["surrogate_comparison"])

    print("Selected features")
    display(result["selected_features"])

    print("Human-readable denormalized rules")
    display(
        result["reporting_rules"][
            [
                "rule_id",
                "if_denormalized",
                "then",
                "training_samples",
                "leaf_purity",
                "rule_length",
            ]
        ]
    )

    print("Saved reports to:", result["report_dir"])
    print("Saved theory to:", result["theory_dir"])

print("\nAll three model experiments completed successfully.")



Running complete experiment: Decision Tree

Black-box test metrics


,value
n_samples,14302.000000
prevalence,0.087960
predicted_positive_rate,0.486925
accuracy,0.539085
balanced_accuracy,0.588225
precision,0.117030
recall_sensitivity,0.647854
specificity,0.528596
f1,0.198249
f2,0.339697


,predicted 0,predicted 1
actual 0,6895,6149
actual 1,443,815


Safety verification


,value
n_rows_checked,15172
n_deceased_rows,1084
n_non_deceased_rows,14088
all_invariants_passed,True


...

Selected features


,feature,standardized
0,glipizide,False
1,glyburide,False
2,discharge_disposition_id_1,False
3,discharge_disposition_id_5,False
4,discharge_disposition_id_11,False
5,discharge_disposition_id_22,False
6,num_lab_procedures,True
7,number_inpatient,True


Human-readable denormalized rules


,rule_id,if_denormalized,then,training_samples,leaf_purity,rule_length
0,1,previous inpatient visits (binned) is 0 AND di...,not_readmitted,5382,0.991825,3
1,2,previous inpatient visits (binned) is 0 AND di...,not_readmitted,93,0.655914,3
2,3,previous inpatient visits (binned) is 1 AND di...,readmitted,451,0.614191,3
3,4,previous inpatient visits (binned) is 1 AND di...,not_readmitted,311,0.967846,3
4,5,previous inpatient visits (binned) is one of {...,readmitted,46,0.760870,4
5,6,previous inpatient visits (binned) is one of {...,not_readmitted,8,0.875000,4
6,7,previous inpatient visits (binned) is one of {...,readmitted,533,0.900563,3
7,8,previous inpatient visits (binned) is 2 AND di...,readmitted,342,0.932749,3
8,9,previous inpatient visits (binned) is 2 AND di...,not_readmitted,6,1.000000,3
9,10,previous inpatient visits (binned) is 2 AND di...,readmitted,176,0.710227,3


Saved reports to: C:\Users\ACER-PC\Desktop\Projects\PythonProjects\trustworthy_clinical_cds\reports\safety_symbolic_model_analysis\mlp
Saved theory to: C:\Users\ACER-PC\Desktop\Projects\PythonProjects\trustworthy_clinical_cds\theories\safety_symbolic_model_analysis\mlp

All three model experiments completed successfully.


### Critical interpretation of the main results

True positive prevalence is 8.8%, whereas Decision Tree and XGBoost predict the positive class much more frequently. This improves sensitivity at the cost of many false alerts.

#### Decision Tree

- Recall is 0.648 but precision is 0.117: 815 true positives are accompanied by 6,149 false positives. Its 48.7% predicted-positive rate is far above prevalence.
- PSyKE covers the complete test set and reaches 0.971 overall fidelity. Class-conditional fidelity is balanced: 0.974 for black-box `not_readmitted` decisions and 0.968 for `readmitted` decisions.
- The constant rule reaches only 0.513 fidelity, so the twelve-rule surrogate provides a substantial improvement.
- Several discharge-disposition features are selected. Some leaves have very limited support or moderate purity and require cautious rule-level interpretation.

#### XGBoost

- Recall is 0.608 and precision is 0.122, with 765 true positives and 5,488 false positives. Positive predictions again greatly exceed observed prevalence.
- The surrogate reaches full coverage and 0.965 overall fidelity. Class-conditional fidelity is 0.947 for `not_readmitted` and 0.989 for `readmitted`.
- The constant rule reaches 0.563 fidelity, approximately 0.403 below PSyKE.
- A few extracted leaves have very small support or purity around 0.56–0.65. High global fidelity does not make every individual rule equally reliable.

#### MLP

- The MLP is substantially more conservative: it predicts positive for 3.38% of test rows and reaches only 0.102 recall, with 1,130 false negatives and 128 true positives. Its 0.896 accuracy is largely driven by imbalance; balanced accuracy is 0.537.
- PSyKE reaches full coverage and 0.946 fidelity, with similar fidelity for both black-box classes (0.947 and 0.942).
- The constant `not_readmitted` rule reaches higher aggregate fidelity, 0.966, because the black box itself predicts class 0 almost everywhere. That constant rule nevertheless has zero fidelity for black-box `readmitted` decisions, whereas PSyKE reproduces approximately 94.2% of them.
- This model demonstrates why aggregate fidelity must always be accompanied by class-conditional fidelity.

#### Discharge-disposition features

`discharge_disposition_id_*` variables occur in all three complete surrogates, and `discharge_disposition_id_1` is selected for every model. Its repeated appearance indicates a strong association with learned decisions, but does not establish causality or clinical coherence. Because discharge disposition may also conflict with the declared decision time, its explanatory importance must be reported as a temporal-validity concern.

## 8. Compact cross-model comparison

The table combines the main quantities while keeping task performance, safety, and symbolic fidelity in separate columns.

In [18]:
experiment_summary = pd.DataFrame(
    [result["summary"] for result in experiment_results.values()]
).set_index("model_key")

display(experiment_summary)
experiment_summary.to_csv(OUTPUT_ROOT / "all_models_summary.csv")
normalization_table.to_csv(OUTPUT_ROOT / "normalization_registry.csv")

write_json(
    {
        "models": list(MODEL_SPECS),
        "data_artifacts": {
            split_name: {
                key: (path.name if path is not None else None)
                for key, path in paths.items()
            }
            for split_name, paths in split_paths.items()
        },
        "normalization_source": normalization_source,
        "safety_context_source": safety_context_source,
        "psyke_config": PSYKE_CONFIG,
        "output_root": OUTPUT_ROOT,
        "theory_root": THEORY_ROOT,
    },
    OUTPUT_ROOT / "experiment_manifest.json",
)

print("Combined summary:", OUTPUT_ROOT / "all_models_summary.csv")


,model,threshold,n_features,test_recall,test_precision,test_f2,test_roc_auc,test_average_precision,safety_deceased_rows,safety_passed,symbolic_coverage,symbolic_fidelity_overall,fidelity_not_readmitted,fidelity_readmitted,trivial_fidelity,fidelity_gain_over_trivial,n_rules,tree_depth,n_selected_features
model_key,,,,,,,,,,,,,,,,,,,
decision_tree,Decision Tree,0.500000,148,0.647854,0.117030,0.339697,0.639540,0.153615,1084,True,1.0,0.971333,0.974380,0.968122,0.513075,0.458258,12,4,9
xgboost,XGBoost,0.500000,148,0.608108,0.122341,0.338946,0.643284,0.163851,1084,True,1.0,0.965459,0.946826,0.989445,0.562788,0.402671,12,4,10
mlp,MLP,0.168384,148,0.101749,0.264463,0.116026,0.648515,0.162064,1084,True,1.0,0.946441,0.946591,0.942149,0.966159,-0.019718,12,4,8


Combined summary: C:\Users\ACER-PC\Desktop\Projects\PythonProjects\trustworthy_clinical_cds\reports\safety_symbolic_model_analysis\all_models_summary.csv


### Cross-model summary

Decision Tree and XGBoost show a clear PSyKE advantage over the constant rule and high fidelity for both black-box classes. The MLP shows the opposite aggregate pattern: the constant rule appears highly faithful only because the MLP is strongly oriented toward `not_readmitted`.

The dummy rule always obtains approximately 0.912 task accuracy because 91.2% of outcomes are negative, but its fidelity differs markedly across models: about 0.513 for Decision Tree, 0.563 for XGBoost, and 0.966 for MLP. It is therefore not generally highly faithful; it is mainly highly faithful to the conservative MLP.

## 9. Actual effect of the safety wrapper

The wrapper neither retrains the models nor changes the preceding PSyKE analyses. It is a post-processing layer: deceased encounters receive final class 0 and readmission probability 0, while every non-deceased output must remain unchanged.

In [19]:
for model_key, result in experiment_results.items():
    print(model_key)
    display(safety_impact_summary(result["safety_audit"]).to_frame())

decision_tree


,value
deceased rows,1084.000000
deceased predicted readmitted before wrapper,7.000000
predictions actually changed,7.000000
probabilities actually changed,7.000000
non-deceased predictions changed,0.000000
non-deceased probabilities changed,0.000000
maximum base probability among deceased,0.509112
maximum final probability among deceased,0.000000


xgboost


,value
deceased rows,1084.000000
deceased predicted readmitted before wrapper,7.000000
predictions actually changed,7.000000
probabilities actually changed,1084.000000
non-deceased predictions changed,0.000000
non-deceased probabilities changed,0.000000
maximum base probability among deceased,0.528667
maximum final probability among deceased,0.000000


mlp


,value
deceased rows,1084.000000
deceased predicted readmitted before wrapper,1.000000
predictions actually changed,1.000000
probabilities actually changed,1084.000000
non-deceased predictions changed,0.000000
non-deceased probabilities changed,0.000000
maximum base probability among deceased,0.175752
maximum final probability among deceased,0.000000


### Safety-audit interpretation

- Decision Tree and XGBoost classified 7 of the 1,084 deceased encounters as `readmitted`; the MLP classified 1. The wrapper changes exactly those hard decisions.
- XGBoost and MLP assigned a non-zero positive probability to all deceased encounters, even when it remained below threshold, so all 1,084 probabilities are set to zero. For Decision Tree, only seven probabilities change because the others were already zero.
- No prediction or probability changes for the 14,088 non-deceased encounters.

This demonstrates enforcement of the safety constraint, not improved predictive performance. Test metrics describe the thresholded classifier before the wrapper; the audit reports the wrapper's separate operational effect.

## 10. Symbolic ablation of `discharge_disposition_id_1`

This experiment keeps the original 148-feature black box unchanged but prevents the symbolic surrogate from using `discharge_disposition_id_1`. The ablated surrogate uses the same limits as the complete surrogate (`max_depth=4`, `max_leaves=12`), so the comparison does not introduce additional tree capacity.

In [20]:
# PSyKE/CART surrogate ablation: remove discharge_disposition_id_1 from the rules.
#
# The black-box models still receive all 148 original features.
# PSyKE receives the black-box training decisions but can construct rules
# using only the remaining 147 features.

ABLATION_FEATURE = "discharge_disposition_id_1"
ABLATION_PSYKE_CONFIG = dict(PSYKE_CONFIG)

if ABLATION_FEATURE not in model_contract:
    raise KeyError(
        f"{ABLATION_FEATURE!r} is not present in the model feature contract."
    )

X_train_full = split_frames["train"].loc[:, model_contract].copy()
X_test_full = split_frames["test"].loc[:, model_contract].copy()

# These are the 147 features available to the ablated symbolic surrogate.
X_train_without_feature = X_train_full.drop(columns=ABLATION_FEATURE)
X_test_without_feature = X_test_full.drop(columns=ABLATION_FEATURE)

y_test = split_targets["test"]

ablation_results = {}
comparison_rows = []

for model_key, spec in MODEL_SPECS.items():
    print(f"\nAblation experiment: {spec['label']}")

    predictor = ThresholdedClassifier(
        loaded_models[model_key],
        threshold=float(spec["threshold"]),
    )

    # Obtain the original black-box decisions using all 148 features.
    black_box_train_numeric = np.asarray(
        predictor.predict(X_train_full),
        dtype=int,
    )
    black_box_test_numeric = np.asarray(
        predictor.predict(X_test_full),
        dtype=int,
    )

    # PSyKE can query the original training decisions, but receives only
    # the 147 non-ablated features.
    ablation_oracle = PrecomputedLabelOracle(
        X_train_without_feature,
        black_box_train_numeric,
    )

    symbolic_without_feature = extract_psyke_cart(
        ablation_oracle,
        X_train_without_feature,
        **ABLATION_PSYKE_CONFIG,
    )

    # Evaluate the new symbolic theory on the untouched test set.
    test_safe = symbolic_without_feature.feature_map.to_safe(
        X_test_without_feature
    )

    symbolic_test_labels = np.asarray(
        symbolic_without_feature.extractor.predict(test_safe),
        dtype=object,
    )

    black_box_test_labels = np.where(
        black_box_test_numeric == 1,
        "readmitted",
        "not_readmitted",
    ).astype(object)

    ablation_metrics = symbolic_surrogate_metrics(
        black_box_test_labels,
        symbolic_test_labels,
        y_test,
    )

    ablation_classwise = class_conditional_fidelity_table(
        black_box_test_labels,
        symbolic_test_labels,
    )

    selected_features = selected_rule_features(
        symbolic_without_feature
    )

    assert ABLATION_FEATURE not in selected_features

    ablation_rules = denormalized_rule_table(
        symbolic_without_feature,
        normalization_registry,
    )

    full_metrics = experiment_results[model_key]["symbolic_metrics"]
    full_classwise = experiment_results[model_key]["classwise"]

    comparison_rows.extend([
        {
            "model": spec["label"],
            "surrogate": "Complete PSyKE",
            "fidelity_overall": full_metrics["fidelity_overall"],
            "fidelity_not_readmitted": full_classwise.loc[
                "not_readmitted", "class_conditional_fidelity"
            ],
            "fidelity_readmitted": full_classwise.loc[
                "readmitted", "class_conditional_fidelity"
            ],
            "coverage": full_metrics["coverage"],
            "n_rules": full_metrics["n_rules"],
            "n_selected_features": full_metrics[
                "n_distinct_features_used"
            ],
        },
        {
            "model": spec["label"],
            "surrogate": f"Without {ABLATION_FEATURE}",
            "fidelity_overall": ablation_metrics["fidelity_overall"],
            "fidelity_not_readmitted": ablation_classwise.loc[
                "not_readmitted", "class_conditional_fidelity"
            ],
            "fidelity_readmitted": ablation_classwise.loc[
                "readmitted", "class_conditional_fidelity"
            ],
            "coverage": ablation_metrics["coverage"],
            "n_rules": symbolic_without_feature.n_rules,
            "n_selected_features": len(selected_features),
        },
    ])

    ablation_results[model_key] = {
        "symbolic": symbolic_without_feature,
        "metrics": ablation_metrics,
        "classwise": ablation_classwise,
        "rules": ablation_rules,
        "selected_features": selected_features,
    }

    print("Selected features:", selected_features)
    display(ablation_classwise)

comparison = pd.DataFrame(comparison_rows)

# Calculate the fidelity difference against the complete surrogate.
comparison["fidelity_change_vs_complete"] = (
    comparison.groupby("model")["fidelity_overall"].transform(
        lambda values: values - values.iloc[0]
    )
)

display(
    comparison.set_index(["model", "surrogate"])
)

comparison.to_csv(
    OUTPUT_ROOT / "ablation_without_discharge_disposition_id_1.csv",
    index=False,
)


Ablation experiment: Decision Tree
Selected features: ('age', 'discharge_disposition_id_3', 'discharge_disposition_id_6', 'discharge_disposition_id_11', 'time_in_hospital', 'num_lab_procedures', 'number_inpatient', 'number_diagnoses')


,n_black_box_predictions,symbolic_coverage,class_conditional_fidelity
black_box_class,,,
not_readmitted,7338,1.0,0.975061
readmitted,6964,1.0,0.791356



Ablation experiment: XGBoost
Selected features: ('discharge_disposition_id_3', 'discharge_disposition_id_6', 'discharge_disposition_id_18', 'num_lab_procedures', 'number_inpatient')


,n_black_box_predictions,symbolic_coverage,class_conditional_fidelity
black_box_class,,,
not_readmitted,8049,1.0,0.976767
readmitted,6253,1.0,0.775148



Ablation experiment: MLP
Selected features: ('glipizide', 'A1Cresult', 'discharge_disposition_id_3', 'discharge_disposition_id_5', 'discharge_disposition_id_11', 'discharge_disposition_id_22', 'num_lab_procedures', 'number_inpatient')


,n_black_box_predictions,symbolic_coverage,class_conditional_fidelity
black_box_class,,,
not_readmitted,13818,1.0,0.965263
readmitted,484,1.0,0.855372


fidelity_overall  \
model         surrogate                                              
Decision Tree Complete PSyKE                              0.971333   
              Without discharge_disposition_id_1          0.885610   
XGBoost       Complete PSyKE                              0.965459   
              Without discharge_disposition_id_1          0.888617   
MLP           Complete PSyKE                              0.946441   
              Without discharge_disposition_id_1          0.961544   

                                                  fidelity_not_readmitted  \
model         surrogate                                                     
Decision Tree Complete PSyKE                                     0.974380   
              Without discharge_disposition_id_1                 0.975061   
XGBoost       Complete PSyKE                                     0.946826   
              Without discharge_disposition_id_1                 0.976767   
MLP           Complete PSyKE                                     0.946591   
              Without discharge_disposition_id_1                 0.965263   

                                                  fidelity_readmitted  \
model         surrogate                                                 
Decision Tree Complete PSyKE                                 0.968122   
              Without discharge_disposition_id_1             0.791356   
XGBoost       Complete PSyKE                                 0.989445   
              Without discharge_disposition_id_1             0.775148   
MLP           Complete PSyKE                                 0.942149   
              Without discharge_disposition_id_1             0.855372   

                                                  coverage  n_rules  \
model         surrogate                                               
Decision Tree Complete PSyKE                           1.0       12   
              Without discharge_disposition_id_1       1.0       11   
XGBoost       Complete PSyKE                           1.0       12   
              Without discharge_disposition_id_1       1.0        6   
MLP           Complete PSyKE                           1.0       12   
              Without discharge_disposition_id_1       1.0       12   

                                                  n_selected_features  \
model         surrogate                                                 
Decision Tree Complete PSyKE                                        9   
              Without discharge_disposition_id_1                    8   
XGBoost       Complete PSyKE                                       10   
              Without discharge_disposition_id_1                    5   
MLP           Complete PSyKE                                        8   
              Without discharge_disposition_id_1                    8   

                                                  fidelity_change_vs_complete  
model         surrogate                                                        
Decision Tree Complete PSyKE                                         0.000000  
              Without discharge_disposition_id_1                    -0.085722  
XGBoost       Complete PSyKE                                         0.000000  
              Without discharge_disposition_id_1                    -0.076842  
MLP           Complete PSyKE                                         0.000000  
              Without discharge_disposition_id_1                     0.015103

### How to interpret the updated ablation

The ablation concerns the **symbolic explanation**, not model retraining: every black box still receives all 148 features, while PSyKE must imitate its stored decisions using the remaining 147 explanatory inputs.

Any loss of overall or class-conditional fidelity therefore measures how difficult it is to express the existing black-box decisions without this feature under the same rule budget. It still does not establish a causal effect, a fairness bias, or the performance of a model retrained without the feature.

## 11. Single-feature benchmark using `discharge_disposition_id_1`

The final experiment trains a small intermediate CART using only the one-hot feature `discharge_disposition_id_1`, then extracts a PSyKE theory from it. Comparison with the complete black box measures how much of the decision boundary can be reproduced by this single split.

The identifier is kept as a code. Its semantic description is intentionally deferred to the report.

In [22]:
# Single-feature symbolic benchmark:
# imitate each complete black box using only discharge_disposition_id_1.

from sklearn.tree import DecisionTreeClassifier

from src.trustworthy_cds.symbolic import balanced_oracle_sample


BENCHMARK_FEATURE = "discharge_disposition_id_1"

SINGLE_FEATURE_PSYKE_CONFIG = {
    "max_samples": PSYKE_CONFIG["max_samples"],
    "max_depth": 3,
    "max_leaves": 4,
    "random_state": PSYKE_CONFIG["random_state"],
}

if BENCHMARK_FEATURE not in model_contract:
    raise KeyError(
        f"{BENCHMARK_FEATURE!r} is not present in the model feature contract."
    )

X_train_full = split_frames["train"].loc[:, model_contract].copy()
X_test_full = split_frames["test"].loc[:, model_contract].copy()

X_train_single = X_train_full[[BENCHMARK_FEATURE]]
X_test_single = X_test_full[[BENCHMARK_FEATURE]]

y_test = split_targets["test"]

single_feature_results = {}
comparison_rows = []

benchmark_output_dir = (
    OUTPUT_ROOT / "single_feature_discharge_disposition_id_1"
)
benchmark_output_dir.mkdir(parents=True, exist_ok=True)

for model_key, spec in MODEL_SPECS.items():
    print("\n" + "=" * 80)
    print(f"Single-feature benchmark: {spec['label']}")
    print("=" * 80)

    predictor = ThresholdedClassifier(
        loaded_models[model_key],
        threshold=float(spec["threshold"]),
    )

    # Create the same type of balanced extraction sample used by the
    # complete symbolic experiment. Labels come from the complete black box.
    extraction_sample_full = balanced_oracle_sample(
        X_train_full,
        predictor,
        max_samples=PSYKE_CONFIG["max_samples"],
        random_state=PSYKE_CONFIG["random_state"],
    )

    extraction_labels = np.asarray(
        predictor.predict(extraction_sample_full),
        dtype=int,
    )

    # This compact intermediate CART learns the black-box decisions
    # using only the selected benchmark feature.
    one_feature_cart = DecisionTreeClassifier(
        max_depth=3,
        max_leaf_nodes=4,
        random_state=PSYKE_CONFIG["random_state"],
    )

    one_feature_cart.fit(
        extraction_sample_full[[BENCHMARK_FEATURE]],
        extraction_labels,
    )

    # PSyKE converts the one-feature CART behaviour into a logical theory.
    try:
        symbolic_single_feature = extract_psyke_cart(
            one_feature_cart,
            X_train_single,
            **SINGLE_FEATURE_PSYKE_CONFIG,
        )
    except ValueError as exc:
        if "produced only one class" not in str(exc):
            raise
        single_feature_results[model_key] = {
            "status": "non_informative_single_class",
            "intermediate_cart": one_feature_cart,
            "error": str(exc),
        }
        print(
            "No informative PSyKE theory was extracted: the one-feature "
            "predictor returned a single class on the extraction pool."
        )
        continue

    # Original black-box decisions on the untouched test set.
    black_box_test_numeric = np.asarray(
        predictor.predict(X_test_full),
        dtype=int,
    )

    black_box_test_labels = np.where(
        black_box_test_numeric == 1,
        "readmitted",
        "not_readmitted",
    ).astype(object)

    # Decisions of the intermediate one-feature CART.
    single_cart_test_numeric = np.asarray(
        one_feature_cart.predict(X_test_single),
        dtype=int,
    )

    single_cart_test_labels = np.where(
        single_cart_test_numeric == 1,
        "readmitted",
        "not_readmitted",
    ).astype(object)

    # Final predictions made by the PSyKE theory.
    test_safe = symbolic_single_feature.feature_map.to_safe(
        X_test_single
    )

    symbolic_test_labels = np.asarray(
        symbolic_single_feature.extractor.predict(test_safe),
        dtype=object,
    )

    # Main result: PSyKE one-feature theory versus complete black box.
    benchmark_metrics = symbolic_surrogate_metrics(
        black_box_test_labels,
        symbolic_test_labels,
        y_test,
    )

    benchmark_classwise = class_conditional_fidelity_table(
        black_box_test_labels,
        symbolic_test_labels,
    )

    # Internal check: PSyKE should faithfully reproduce the intermediate
    # one-feature CART from which its theory was extracted.
    psyke_to_single_cart = symbolic_surrogate_metrics(
        single_cart_test_labels,
        symbolic_test_labels,
    )

    selected_features = selected_rule_features(
        symbolic_single_feature
    )

    if set(selected_features) - {BENCHMARK_FEATURE}:
        raise AssertionError(
            "The single-feature theory unexpectedly selected other features."
        )

    readable_rules = denormalized_rule_table(
        symbolic_single_feature,
        normalization_registry,
    )

    full_metrics = experiment_results[model_key]["symbolic_metrics"]
    full_classwise = experiment_results[model_key]["classwise"]

    full_balanced_fidelity = float(
        full_classwise["class_conditional_fidelity"].mean()
    )
    benchmark_balanced_fidelity = float(
        benchmark_classwise["class_conditional_fidelity"].mean()
    )

    comparison_rows.append({
        "model": spec["label"],
        "complete_psyke_fidelity": full_metrics["fidelity_overall"],
        "discharge_disposition_id_1_only_fidelity": (
            benchmark_metrics["fidelity_overall"]
        ),
        "fidelity_change": (
            benchmark_metrics["fidelity_overall"]
            - full_metrics["fidelity_overall"]
        ),
        "complete_balanced_fidelity": full_balanced_fidelity,
        "discharge_disposition_id_1_only_balanced_fidelity": (
            benchmark_balanced_fidelity
        ),
        "balanced_fidelity_change": (
            benchmark_balanced_fidelity
            - full_balanced_fidelity
        ),
        "fidelity_not_readmitted": benchmark_classwise.loc[
            "not_readmitted",
            "class_conditional_fidelity",
        ],
        "fidelity_readmitted": benchmark_classwise.loc[
            "readmitted",
            "class_conditional_fidelity",
        ],
        "coverage": benchmark_metrics["coverage"],
        "n_rules": symbolic_single_feature.n_rules,
        "psyke_fidelity_to_single_feature_cart": (
            psyke_to_single_cart["fidelity_overall"]
        ),
    })

    single_feature_results[model_key] = {
        "intermediate_cart": one_feature_cart,
        "symbolic": symbolic_single_feature,
        "metrics": benchmark_metrics,
        "classwise": benchmark_classwise,
        "rules": readable_rules,
    }

    readable_rules.to_csv(
        benchmark_output_dir / f"{model_key}_rules.csv",
        index=False,
    )

    print("Class-conditional fidelity against the complete black box:")
    display(benchmark_classwise)

    print(f"Rules based only on {BENCHMARK_FEATURE}:")
    display(
        readable_rules[
            [
                "rule_id",
                "if_denormalized",
                "then",
                "training_samples",
                "leaf_purity",
            ]
        ]
    )

single_feature_comparison = pd.DataFrame(comparison_rows).set_index(
    "model"
)

display(single_feature_comparison)

single_feature_comparison.to_csv(
    benchmark_output_dir / "single_feature_comparison.csv"
)


Single-feature benchmark: Decision Tree
Class-conditional fidelity against the complete black box:


,n_black_box_predictions,symbolic_coverage,class_conditional_fidelity
black_box_class,,,
not_readmitted,7338,1.0,0.967021
readmitted,6964,1.0,0.737220


Rules based only on discharge_disposition_id_1:


,rule_id,if_denormalized,then,training_samples,leaf_purity
0,1,discharge disposition code is not 1,readmitted,4000,1.0
1,2,discharge disposition code is 1,not_readmitted,4000,1.0



Single-feature benchmark: XGBoost
Class-conditional fidelity against the complete black box:


,n_black_box_predictions,symbolic_coverage,class_conditional_fidelity
black_box_class,,,
not_readmitted,8049,1.0,0.970680
readmitted,6253,1.0,0.822005


Rules based only on discharge_disposition_id_1:


,rule_id,if_denormalized,then,training_samples,leaf_purity
0,1,discharge disposition code is not 1,readmitted,4000,1.0
1,2,discharge disposition code is 1,not_readmitted,4000,1.0



Single-feature benchmark: MLP
No informative PSyKE theory was extracted: the one-feature predictor returned a single class on the extraction pool.


,complete_psyke_fidelity,discharge_disposition_id_1_only_fidelity,fidelity_change,complete_balanced_fidelity,discharge_disposition_id_1_only_balanced_fidelity,balanced_fidelity_change,fidelity_not_readmitted,fidelity_readmitted,coverage,n_rules,psyke_fidelity_to_single_feature_cart
model,,,,,,,,,,,
Decision Tree,0.971333,0.855125,-0.116208,0.971251,0.852120,-0.119130,0.967021,0.737220,1.0,2,1.0
XGBoost,0.965459,0.905678,-0.059782,0.968135,0.896343,-0.071793,0.970680,0.822005,1.0,2,1.0


### Single-feature benchmark interpretation

For Decision Tree, two rules based only on `discharge_disposition_id_1` reach approximately 0.855 overall fidelity: 0.967 for black-box `not_readmitted` decisions and 0.737 for `readmitted`. For XGBoost, overall fidelity is approximately 0.906, with class-conditional values of 0.971 and 0.822. These are high values for two-rule theories, but remain below the complete surrogates and are less balanced on the positive class.

This confirms that `discharge_disposition_id_1` is a strong decision separator for Decision Tree and XGBoost. It does not show that the feature is uniquely important, causal, or temporally appropriate.

For the MLP, the single-feature intermediate predictor returns only one class on the extraction pool. PSyKE correctly declines to create a non-informative classification theory. This is consistent with the MLP's conservative behavior: a constant negative predictor can obtain high aggregate fidelity without explaining its rare `readmitted` decisions.

## 12. Generated artifacts and interpretation limits

A complete run creates model-specific evidence under:

```text
reports/safety_symbolic_model_analysis/
theories/safety_symbolic_model_analysis/
```

The reports include compact test metrics, confusion matrices, safety audits, overall and class-conditional fidelity, constant-rule comparisons, selected features, and standardized and denormalized rule tables. Additional outputs contain the ablation without `discharge_disposition_id_1` and the single-feature benchmark using the same feature.

The extracted rules describe model behavior. They do not establish medical causality, constitute independently validated clinical knowledge, or guarantee temporal validity. Final interpretation must jointly consider black-box task performance, class imbalance, rule support and purity, feature availability, symbolic fidelity, and safety compliance.